# EDA 07 — Logements Sociaux Finances a Paris
**Source** : Paris Open Data | **Bronze** : `data/bronze/social_housing/` + `data/bronze/logement_sociaux/`

In [ ]:

import os, glob, warnings
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import numpy as np
warnings.filterwarnings("ignore")

PALETTE = {
    "primary":   "#1565C0",
    "secondary": "#E53935",
    "accent":    "#2E7D32",
    "neutral":   "#546E7A",
}

plt.rcParams.update({
    "figure.dpi": 150,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.grid.axis": "y",
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
    "font.family": "sans-serif",
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.framealpha": 0.9,
    "legend.fontsize": 10,
})

BRONZE_ROOT = os.path.abspath("../../data/bronze")
NB_DIR      = os.path.abspath(".")
FIGURES_DIR = os.path.join(NB_DIR, "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

def save_fig(name, source_prefix, tight=True):
    dest = os.path.join(FIGURES_DIR, source_prefix)
    os.makedirs(dest, exist_ok=True)
    path = os.path.join(dest, f"{name}.png")
    if tight:
        plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight", facecolor="white")
    print(f"  Saved -> figures/{source_prefix}/{name}.png")

print("Setup OK — figures ->", FIGURES_DIR)


In [ ]:

def draw_schema(title, columns, source_prefix, filename="schema"):
    n_rows = len(columns)
    fig_h  = max(3.0, 0.48 * n_rows + 1.6)
    fig, ax = plt.subplots(figsize=(13, fig_h))
    ax.axis("off")
    fig.suptitle(title, fontsize=14, fontweight="bold", y=0.99, ha="center")
    headers = ["Colonne", "Type", "Description"]
    col_x   = [0.0, 0.22, 0.37]
    col_w   = [0.22, 0.15, 0.63]
    row_h   = 1.0 / (n_rows + 1)
    for i, (hdr, x, w) in enumerate(zip(headers, col_x, col_w)):
        rect = mpatches.FancyBboxPatch(
            (x, 1 - row_h), w - 0.006, row_h * 0.88,
            boxstyle="round,pad=0.01", linewidth=0.5,
            edgecolor="#90A4AE", facecolor="#1565C0",
            transform=ax.transAxes, clip_on=False)
        ax.add_patch(rect)
        ax.text(x + w/2, 1 - row_h/2, hdr,
                transform=ax.transAxes, ha="center", va="center",
                fontsize=11, fontweight="bold", color="white")
    type_colors = {
        "str":"#1565C0","int":"#2E7D32","float":"#6A1B9A",
        "datetime":"#E65100","bool":"#AD1457","date":"#00695C",
    }
    for ridx, (col_name, col_type, col_desc) in enumerate(columns):
        y_top = 1 - (ridx + 2) * row_h
        bg = "#F5F7FA" if ridx % 2 == 0 else "white"
        for x, w in zip(col_x, col_w):
            rect = mpatches.FancyBboxPatch(
                (x, y_top), w - 0.006, row_h * 0.88,
                boxstyle="round,pad=0.01", linewidth=0.5,
                edgecolor="#CFD8DC", facecolor=bg,
                transform=ax.transAxes, clip_on=False)
            ax.add_patch(rect)
        y_mid = y_top + row_h * 0.44
        tc = type_colors.get(col_type.split("[")[0].lower(), "#37474F")
        ax.text(col_x[0]+col_w[0]/2, y_mid, col_name,
                transform=ax.transAxes, ha="center", va="center",
                fontsize=10, fontweight="bold", color="#1A237E")
        ax.text(col_x[1]+col_w[1]/2, y_mid, col_type,
                transform=ax.transAxes, ha="center", va="center",
                fontsize=9, fontfamily="monospace", color=tc)
        ax.text(col_x[2]+0.01, y_mid, col_desc,
                transform=ax.transAxes, ha="left", va="center",
                fontsize=9.5, color="#37474F")
    plt.subplots_adjust(top=0.92, bottom=0)
    save_fig(filename, source_prefix)
    plt.show()


In [ ]:
PREFIX = "07_social_housing"
draw_schema(
    "Bronze Schema (standardise) — Logements Sociaux",
    [
        ("arrondissement",   "int",      "Numero d'arrondissement (1-20)"),
        ("annee",            "int",      "Annee de financement du programme"),
        ("nombre_logements", "int",      "Nombre de logements finances cette annee"),
        ("ingested_at",      "datetime", "Horodatage UTC d'ingestion"),
    ], PREFIX, "schema_standardise"
)

In [ ]:
draw_schema(
    "Bronze Schema (brut) — Logements Sociaux Finances a Paris",
    [
        ("id_livraison",      "str",   "Identifiant unique du programme"),
        ("adresse_programme", "str",   "Adresse du programme"),
        ("annee",             "str",   "Annee de financement (date)"),
        ("nb_logmt_total",    "int",   "Nombre total de logements du programme"),
        ("nb_plai",           "int",   "Logements PLAI (tres social — ressources tres modestes)"),
        ("nb_plus",           "int",   "Logements PLUS (social — ressources modestes)"),
        ("nb_pluscd",         "int",   "Logements PLUS-CD (conventionnement direct)"),
        ("nb_pls",            "int",   "Logements PLS (intermediaires — ressources moyennes)"),
        ("mode_real",         "str",   "Mode de realisation (construction neuve, rehabilitation, etc.)"),
        ("nature_programme",  "str",   "Type : logement familial, foyer, residence..."),
        ("arrdt",             "int",   "Arrondissement"),
        ("coord_x_l93",       "float", "Coordonnee X Lambert 93"),
        ("coord_y_l93",       "float", "Coordonnee Y Lambert 93"),
    ], PREFIX, "schema_brut"
)

In [ ]:
path_std = f"{BRONZE_ROOT}/social_housing/part-0.parquet"
path_raw = f"{BRONZE_ROOT}/logement_sociaux/logements-sociaux-finances-a-paris.parquet"
df_std = pd.read_parquet(path_std) if os.path.exists(path_std) else pd.DataFrame()
df_raw = pd.read_parquet(path_raw) if os.path.exists(path_raw) else pd.DataFrame()
if not df_raw.empty:
    df_raw["annee_int"] = pd.to_datetime(df_raw["annee"],errors="coerce").dt.year
print(f"Standardise : {df_std.shape}"); print(f"Brut        : {df_raw.shape}")

In [ ]:
if not df_std.empty:
    df_time = df_std.groupby("annee")["nombre_logements"].sum().dropna()
    df_time.index = df_time.index.astype(int); df_time = df_time.sort_index()
    fig, ax = plt.subplots(figsize=(14,5))
    norm = df_time.values / df_time.max()
    ax.bar(df_time.index.astype(str),df_time.values,color=plt.cm.Blues(0.3+norm*0.6),edgecolor="white")
    ax.set_xlabel("Annee"); ax.set_ylabel("Logements finances")
    ax.set_title("Evolution annuelle des logements sociaux finances — Paris")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{int(x):,}"))
    plt.xticks(rotation=45)
    save_fig("evolution_annuelle", PREFIX); plt.show()

In [ ]:
if not df_std.empty:
    arr_total = df_std.groupby("arrondissement")["nombre_logements"].sum().sort_values(ascending=False)
    norm = (arr_total-arr_total.min())/(arr_total.max()-arr_total.min())
    fig, ax = plt.subplots(figsize=(14,6))
    ax.bar(arr_total.index.astype(str),arr_total.values,color=plt.cm.YlGnBu(0.2+norm*0.75),edgecolor="white")
    ax.set_xlabel("Arrondissement"); ax.set_ylabel("Total logements sociaux finances (cumule)")
    ax.set_title("Logements sociaux finances par arrondissement (cumul historique)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{int(x):,}"))
    plt.setp(ax.xaxis.get_majorticklabels(),rotation=45)
    save_fig("logements_par_arrondissement", PREFIX); plt.show()

In [ ]:
if not df_raw.empty:
    by_arr = df_raw.groupby("arrdt")[["nb_plai","nb_plus","nb_pluscd","nb_pls"]].sum().sort_index()
    fig, ax = plt.subplots(figsize=(14,7))
    bot = np.zeros(len(by_arr))
    for col, color, label in [("nb_plai","#E53935","PLAI (tres social)"),("nb_plus","#FF8F00","PLUS (social)"),
                               ("nb_pluscd","#FDD835","PLUS-CD"),("nb_pls","#2E7D32","PLS (intermediaire)")]:
        ax.bar(by_arr.index.astype(str),by_arr[col],bottom=bot,label=label,color=color,edgecolor="white",linewidth=0.5)
        bot += by_arr[col].values
    ax.set_xlabel("Arrondissement"); ax.set_ylabel("Logements")
    ax.set_title("Repartition PLAI / PLUS / PLS par arrondissement")
    ax.legend(loc="upper right"); plt.setp(ax.xaxis.get_majorticklabels(),rotation=45)
    save_fig("mix_financement_arrondissement", PREFIX); plt.show()